In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch diffusers transformers accelerate lpips timm safetensors scikit-image matplotlib

In [ ]:
# Dataset: https://drive.google.com/file/d/1QlAdzrHpfBIOZ6SK78yHF2i1u6tikmBc/view
#
# You do not need to download the dataset directly.
# Open the link above and use the "Add shortcut to Drive" button
# (next to the download icon, top right) to add it to your own Drive
# under a folder named "cs559-project" before running this cell.
!mkdir -p SECOND_dataset/train
!unrar x -r /content/drive/MyDrive/cs559-project/SECOND_train_set.rar -d SECOND_dataset/train/

In [ ]:
# run this cell and the one below if working with a saved checkpoint from the previous training of the model
"""
import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/checkpoint-49.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/checkpoint-49')
"""

In [ ]:
import torch
import torch.nn as nn
from diffusers import UNet2DConditionModel, AutoencoderKL
# if there is a saved checkpoint from before
def load_checkpoint(checkpoint_path, device="cuda"):

    vae = AutoencoderKL.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", subfolder="vae").to(device)
    image_encoder = FrozenDinoV2Encoder().to(device)

    unet = UNet2DConditionModel.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", subfolder="unet")
    with torch.no_grad():
        old_conv = unet.conv_in
        # the 7-channel layer
        new_conv = nn.Conv2d(7, old_conv.out_channels, kernel_size=old_conv.kernel_size, padding=old_conv.padding)
        unet.conv_in = new_conv # replace it

    # Loading the UNet weights from checkpoint
    unet_weight_path = f"{checkpoint_path}/unet/diffusion_pytorch_model.bin"
    if not os.path.exists(unet_weight_path):
         unet_weight_path = f"{checkpoint_path}/unet/diffusion_pytorch_model.safetensors"
         from safetensors.torch import load_file
         unet_state_dict = load_file(unet_weight_path)
    else:
        unet_state_dict = torch.load(unet_weight_path, map_location=device)

    unet.load_state_dict(unet_state_dict)
    unet.to(device)
    unet.eval()

    resampler = ImageResampler(
        dim=1024,
        depth=4,
        dim_head=64,
        heads=16,
        num_queries=8,
        embedding_dim=1024,
        output_dim=unet.config.cross_attention_dim,
    )

    resampler_path = f"{checkpoint_path}/resampler.pt"
    resampler.load_state_dict(torch.load(resampler_path, map_location=device))
    resampler.to(device)
    resampler.eval()

    print("Model loaded")
    return unet, resampler, vae, image_encoder

In [ ]:
import os
import shutil
import random
import torch
import torch.nn.functional as F
import numpy as np
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from scipy import linalg
from torchvision.models import inception_v3
import lpips
DATA_DIR = "./SECOND_dataset"

os.makedirs(os.path.join(DATA_DIR, 'validation'), exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'test'), exist_ok=True)

def split_dataset(source_dir, val_ratio=0.2, test_ratio=0.1):
    categories = os.listdir(source_dir)
    files = [os.path.basename(file) for file in os.listdir(os.path.join(source_dir, categories[0]))]
    random.Random(42).shuffle(files)

    for category in categories:
        category_path = os.path.join(source_dir, category)

        if os.path.isdir(category_path):
            total_files = len(files)
            val_count = int(total_files * val_ratio)
            test_count = int(total_files * test_ratio)
            val_files = files[:val_count]
            test_files = files[val_count:val_count + test_count]
            train_files = files[val_count + test_count:]

            # directories for validation and test splits
            os.makedirs(os.path.join(DATA_DIR, 'validation', category), exist_ok=True)
            os.makedirs(os.path.join(DATA_DIR, 'test', category), exist_ok=True)

            for file_name in val_files:
                shutil.move(os.path.join(category_path, file_name),
                            os.path.join(DATA_DIR, 'validation', category, file_name))

            for file_name in test_files:
                shutil.move(os.path.join(category_path, file_name),
                            os.path.join(DATA_DIR, 'test', category, file_name))

            print(f"Processed category: {category}")

split_dataset(os.path.join(DATA_DIR, "train"), val_ratio=0.2, test_ratio=0.1)

print("Dataset split completed.")


2079
415
207
Processed category: label2
2079
415
207
Processed category: label1
2079
415
207
Processed category: im2
2079
415
207
Processed category: im1
Dataset split completed.


In [ ]:
from types import SimpleNamespace
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image
from accelerate import Accelerator
from accelerate.utils import set_seed
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
from tqdm.auto import tqdm
import numpy as np

def reshape_tensor(x, heads):
    bs, length, width = x.shape
    # (bs, length, width) --> (bs, length, n_heads, dim_per_head)
    x = x.view(bs, length, heads, -1)
    # (bs, length, n_heads, dim_per_head) --> (bs, n_heads, length, dim_per_head)
    x = x.transpose(1, 2)
    # (bs, n_heads, length, dim_per_head) --> (bs*n_heads, length, dim_per_head)
    x = x.reshape(bs, heads, length, -1)
    return x
def FeedForward(dim, mult=4):
    inner_dim = int(dim * mult)
    return nn.Sequential(
        nn.LayerNorm(dim),
        nn.Linear(dim, inner_dim, bias=False),
        nn.GELU(),
        nn.Linear(inner_dim, dim, bias=False),
    )


class FrozenDinoV2Encoder(nn.Module):

    def __init__(self, device="cuda", freeze=True):
        super().__init__()
        self.model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14').to(device)
        self.device = device
        if freeze:
            self.freeze()
        self.image_mean = (
            torch.tensor([0.485, 0.456, 0.406]).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
        )
        self.image_std = (
            torch.tensor([0.229, 0.224, 0.225]).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
        )
        # self.projector = nn.Linear(1536, 768)

    def freeze(self):
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

    def forward(self, image):
        if isinstance(image, list):
            image = torch.cat(image, 0)

        image = (
            image.to(self.device) - self.image_mean.to(self.device)
        ) / self.image_std.to(self.device)
        features = self.model.forward_features(image)
        tokens = features["x_norm_patchtokens"]
        image_features = features["x_norm_clstoken"]
        image_features = image_features.unsqueeze(1)
        hint = torch.cat([image_features, tokens], 1)  # 8,257,1024
        # hint = self.projector(hint)
        return hint

    def encode(self, image):
        return self(image)
class PerceiverAttention(nn.Module):
    def __init__(self, *, dim, dim_head=64, heads=8):
        super().__init__()
        self.scale = dim_head**-0.5
        self.dim_head = dim_head
        self.heads = heads
        inner_dim = dim_head * heads

        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

        self.to_q = nn.Linear(dim, inner_dim, bias=False)
        self.to_kv = nn.Linear(dim, inner_dim * 2, bias=False)
        self.to_out = nn.Linear(inner_dim, dim, bias=False)

    def forward(self, x, latents, use_mask=False):
        n_q = latents.shape[-2]
        n_kv = x.shape[-2]

        x = self.norm1(x)
        latents = self.norm2(latents)

        b, l, d = latents.shape

        q = self.to_q(latents)
        kv_input = torch.cat((x, latents), dim=1)
        k, v = self.to_kv(kv_input).chunk(2, dim=-1)

        q = reshape_tensor(q, self.heads)
        k = reshape_tensor(k, self.heads)
        v = reshape_tensor(v, self.heads)

        # attention
        scale = 1 / math.sqrt(math.sqrt(self.dim_head))
        weight = (q * scale) @ (k * scale).transpose(
            -2, -1
        )

        # if use_mask:
        #     block = torch.ones((n_q, n_q + n_kv))
        #     blocks = [block] * x.size(0)
        #     mask = torch.block_diag(*blocks)
        #     additional_rows = torch.ones((n_q, mask.size(1)))
        #     mask = torch.cat((mask, additional_rows), dim=0).unsqueeze(0)
        #     # mask = rearrange(mask, 'b ... -> b (...)')
        #     max_neg_value = -torch.finfo(weight.dtype).max
        #     # mask = repeat(mask, 'b j -> (b h) () j', h=h)
        #     mask = (
        #         mask.repeat(self.heads, 1, 1)
        #         .unsqueeze(0)
        #         .to(torch.bool)
        #         .to(weight.device)
        #     )
        #     weight.masked_fill_(~mask, max_neg_value)

        weight = torch.softmax(weight.float(), dim=-1).type(weight.dtype)
        out = weight @ v

        out = out.permute(0, 2, 1, 3).reshape(b, l, -1)

        return self.to_out(out)

def log_validation(unet, resampler, vae, image_encoder, scheduler, dataloader, accelerator, epoch, step, output_dir, img_size):
    print(f"Running validation at epoch {epoch} step {step}...")
    unet.eval()
    resampler.eval()

    batch = next(iter(dataloader))

    pre_imgs = batch["pre"].to(accelerator.device)
    sem_imgs = batch["sem"].to(accelerator.device)
    post_imgs = batch["post"].to(accelerator.device)

    # encoding pre -image
    pre_imgs_01 = (pre_imgs + 1.0) / 2.0
    with torch.no_grad():
        pre_features = image_encoder(pre_imgs_01)
        encoder_hidden_states = resampler(pre_features)

    # latents
    latent_size = img_size // 8
    latents = torch.randn(
        (pre_imgs.shape[0], 4, latent_size, latent_size),
        device=accelerator.device,
        dtype=unet.dtype
    )

    scheduler.set_timesteps(50)
    # denoising
    for t in tqdm(scheduler.timesteps, disable=True):
        sem_latents = F.interpolate(sem_imgs, size=latents.shape[-2:], mode="nearest")

        unet_input = torch.cat([latents, sem_latents], dim=1)

        with torch.no_grad():
            noise_pred = unet(unet_input, t, encoder_hidden_states=encoder_hidden_states).sample

        latents = scheduler.step(noise_pred, t, latents).prev_sample

    # decoding
    latents = latents / vae.config.scaling_factor
    with torch.no_grad():
        images = vae.decode(latents).sample

    images = (images / 2 + 0.5).clamp(0, 1)
    pre_imgs = (pre_imgs / 2 + 0.5).clamp(0, 1)
    post_imgs = (post_imgs / 2 + 0.5).clamp(0, 1)
    sem_imgs = (sem_imgs / 2 + 0.5).clamp(0, 1)

    images = images.cpu().permute(0, 2, 3, 1).numpy()
    pre_imgs = pre_imgs.cpu().permute(0, 2, 3, 1).numpy()
    post_imgs = post_imgs.cpu().permute(0, 2, 3, 1).numpy()
    sem_imgs = sem_imgs.cpu().permute(0, 2, 3, 1).numpy()

    save_dir = os.path.join(output_dir, "validation")
    os.makedirs(save_dir, exist_ok=True)

    for i in range(images.shape[0]):
        # horizontal concatenation
        combined = np.concatenate([pre_imgs[i], sem_imgs[i], post_imgs[i], images[i]], axis=1)
        combined_img = Image.fromarray((combined * 255).astype(np.uint8))
        combined_img.save(os.path.join(save_dir, f"epoch_{epoch}_step_{step}_sample_{i}.png"))

    unet.train()
    resampler.train()


class SECONDDataset(Dataset):
    def __init__(self, root_dir, split, img_size):
        self.root_dir = root_dir
        self.split = split
        self.img_size = img_size
        self.im1_dir = os.path.join(root_dir, split, "im1")
        self.im2_dir = os.path.join(root_dir, split, "im2")
        self.label_dir = os.path.join(root_dir, split, "label2")  # Change map

        self.files = sorted(os.listdir(self.im1_dir))

    def __len__(self):
        return len(self.files)
    def __getitem__(self, index):
        def load_image(path: str) -> torch.Tensor:
            return TF.to_tensor(Image.open(path).convert("RGB"))

        def normalize(img: torch.Tensor) -> torch.Tensor:
            return TF.normalize(img, [0.5] * 3, [0.5] * 3)

        filename = self.files[index]

        pre_path = os.path.join(self.im1_dir, filename)
        post_path = os.path.join(self.im2_dir, filename)
        sem_path = os.path.join(self.label_dir, filename)

        imgs = (
            load_image(pre_path),
            load_image(post_path),
            load_image(sem_path)
        )

        pre_img = TF.resize(imgs[0], (self.img_size, self.img_size), interpolation=TF.InterpolationMode.BILINEAR)
        post_img = TF.resize(imgs[1], (self.img_size, self.img_size), interpolation=TF.InterpolationMode.BILINEAR)

        sem_img = TF.resize(imgs[2], (self.img_size, self.img_size), interpolation=TF.InterpolationMode.NEAREST)

        pre_img = normalize(pre_img)
        post_img = normalize(post_img)
        sem_img = normalize(sem_img)

        return {"pre": pre_img, "post": post_img, "sem": sem_img}

class ImageResampler(nn.Module):
    """
    Modifies the feature embeddings from dinov2 so they can be adaptable to the cross-attention layer (our proposed contribution)
    """

    def __init__(
        self,
        dim=1024,
        depth=4,
        dim_head=64,
        heads=16,
        num_queries=8,
        embedding_dim=1024,
        output_dim=768,
        ff_mult=4,
    ):
        super().__init__()
        self.latents = nn.Parameter(torch.randn(1, num_queries, dim) / dim**0.5)

        self.proj_in = nn.Linear(embedding_dim, dim)
        self.proj_out = nn.Linear(dim, output_dim)
        self.norm_out = nn.LayerNorm(output_dim)

        self.layers = nn.ModuleList([])
        for _ in range(depth):
            self.layers.append(
                nn.ModuleList(
                    [
                        PerceiverAttention(dim=dim, dim_head=dim_head, heads=heads),
                        FeedForward(dim=dim, mult=ff_mult),
                    ]
                )
            )

    def forward(self, x):
        x = self.proj_in(x)

        latents = self.latents.repeat(x.size(0), 1, 1)

        for attn, ff in self.layers:
            # x is the context (image features), latents is the query
            latents = attn(x, latents) + latents
            latents = ff(latents) + latents

        latents = self.proj_out(latents)
        return self.norm_out(latents)


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
    TrainingArguments,
    Trainer,
)
import evaluate
import torch.nn.functional as F
from scipy import linalg
from torchvision.models import inception_v3
import lpips
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from torchmetrics import JaccardIndex

"""
Finetuning the segment model for our 6 classed dataset to more accurately calculate mIOU
"""

MODEL_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"
DATA_DIR = "./SECOND_dataset"
OUTPUT_DIR = "./segformer_finetuned_change_bridge"
NUM_EPOCHS = 10
BATCH_SIZE = 8
LEARNING_RATE = 0.00006
NUM_CLASSES = 7

class SemanticSegmentationDataset(Dataset):
    def __init__(self, root_dir, processor, split="train"):
        self.root_dir = root_dir
        self.split = split
        self.processor = processor

        self.img_dir = os.path.join(root_dir, split, "im2")
        self.mask_dir = os.path.join(root_dir, split, "label2")

        self.images = sorted(os.listdir(self.img_dir))
        self.masks = sorted(os.listdir(self.mask_dir))

        # Our dataset classes palette
        self.palette = {
            (255, 255, 255): 0,  # Non-change (White)
            (128, 128, 128): 1,  # N.v.g surface (Grey)
            (0, 128, 0):     2,  # Low vegetation (Dark Green)
            (0, 255, 0):     3,  # Tree (Light Green)
            (0, 0, 255):     4,  # Water (Blue)
            (128, 0, 0):     5,  # Building (Dark Red/Brown)
            (255, 0, 0):     6   # Playground (Red)
        }

    def __len__(self):
        return len(self.images)

    def rgb_to_mask(self, rgb_image):
        arr = np.array(rgb_image)
        mask = np.zeros(arr.shape[:2], dtype=np.uint8)

        for color, class_id in self.palette.items():
            matches = np.all(arr == color, axis=-1)
            mask[matches] = class_id

        return mask

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        image = Image.open(img_path).convert("RGB")
        mask_rgb = Image.open(mask_path).convert("RGB")

        # RGB Mask -> Class ID Mask (0-6)
        segmentation_map = self.rgb_to_mask(mask_rgb)

        # normalization, resizing, etc.
        encoding = self.processor(images=image, segmentation_maps=Image.fromarray(segmentation_map), return_tensors="pt")

        for k, v in encoding.items():
            encoding[k] = v.squeeze()

        return encoding


processor = SegformerImageProcessor.from_pretrained(MODEL_NAME)
train_ds = SemanticSegmentationDataset(DATA_DIR, processor, split="train")
val_ds = SemanticSegmentationDataset(DATA_DIR, processor, split="validation" if os.path.exists(os.path.join(DATA_DIR, "validation")) else "train")

metric = evaluate.load("mean_iou")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits_tensor = torch.from_numpy(logits)
    logits_tensor = torch.nn.functional.interpolate(
        logits_tensor,
        size=labels.shape[-2:],
        mode="bilinear",
        align_corners=False,
    ).argmax(dim=1)

    pred_labels = logits_tensor.detach().cpu().numpy()
    metrics = metric.compute(predictions=pred_labels, references=labels, num_labels=NUM_CLASSES, ignore_index=255)
    return {"mean_iou": metrics["mean_iou"], "mean_accuracy": metrics["mean_accuracy"]}

model = SegformerForSemanticSegmentation.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    save_total_limit=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    remove_unused_columns=False,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:417: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b0-finetuned-ade-512-512 and are newly initialized because the shapes did not match:
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([7]) in the model instantiated
- decode_head.classifier.weight: found shape torch.Size([150, 256, 1, 1]) in the checkpoint and torch.Size([7, 256, 1, 1]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

 3


wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss,Mean Iou,Mean Accuracy
1,0.721800,0.709055,0.113806,0.142862
2,0.655600,0.634021,0.125408,0.153984
3,0.619300,0.615582,0.134508,0.163119
4,0.524000,0.623697,0.142194,0.171403
5,0.643900,0.613186,0.144886,0.174014
6,0.581600,0.613706,0.157028,0.189598
7,0.538600,0.621686,0.162689,0.196269
8,0.517600,0.624035,0.155573,0.186929
9,0.517400,0.628224,0.162383,0.195791
10,0.519200,0.635855,0.158450,0.190314


/usr/local/lib/python3.12/dist-packages/datasets/features/image.py:357: UserWarning: Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")
/usr/local/lib/python3.12/dist-packages/datasets/features/image.py:357: UserWarning: Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")
/usr/local/lib/python3.12/dist-packages/datasets/features/image.py:357: UserWarning: Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible with 'Pillow'")
/usr/local/lib/python3.12/dist-packages/datasets/features/image.py:357: UserWarning: Downcasting array dtype int64 to int32 to be compatible with 'Pillow'
  warnings.warn(f"Downcasting array dtype {dtype} to {dest_dtype} to be compatible 

Model saved to ./segformer_finetuned_change_bridge


In [ ]:
import os
import json
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from accelerate import Accelerator
from torch.utils.data import DataLoader
from diffusers import DDPMScheduler, AutoencoderKL, UNet2DConditionModel

from scipy import linalg
from torchvision.models import inception_v3
import lpips
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
from torchmetrics import JaccardIndex

class EvaluationMetrics:
    def __init__(self, device='cuda', num_classes=7, seg_model_id="./segformer_finetuned_change_bridge"):
        self.device = device
        self.num_classes = num_classes

        self.palette_map = {
            (255, 255, 255): 0,  # Non-change
            (128, 128, 128): 1,  # N.v.g surface
            (0, 128, 0):     2,  # Low vegetation
            (0, 255, 0):     3,  # Tree
            (0, 0, 255):     4,  # Water
            (128, 0, 0):     5,  # Building
            (255, 0, 0):     6   # Playground
        }

        self.lpips_fn = lpips.LPIPS(net='alex').to(device).eval()
        self.inception = inception_v3(pretrained=True, transform_input=False).to(device).eval()
        self.miou_metric = JaccardIndex(task="multiclass", num_classes=num_classes).to(device)

        print(f"Loading SegFormer model: {seg_model_id}")
        try:
            self.seg_processor = SegformerImageProcessor.from_pretrained(seg_model_id)
            self.seg_model = SegformerForSemanticSegmentation.from_pretrained(
                seg_model_id,
                num_labels=num_classes,
                ignore_mismatched_sizes=True
            ).to(device).eval()
        except Exception as e:
            print(f"Failed to load SegFormer. Error: {e}")
            self.seg_model = None

    def _rgb_to_sem_id(self, sem_tensor):
        sem_np = (sem_tensor.permute(0, 2, 3, 1).detach().cpu().numpy() * 255).astype(np.uint8)
        batch_size, h, w, _ = sem_np.shape
        mask_id = np.zeros((batch_size, h, w), dtype=np.int64)

        for color, class_id in self.palette_map.items():
            matches = np.all(sem_np == color, axis=-1)
            mask_id[matches] = class_id

        return torch.from_numpy(mask_id).to(self.device)

    def compute_miou(self, generated, ground_truth_sem):
        if self.seg_model is None:
            return 0.0

        ground_truth_sem = ground_truth_sem.to(self.device)
        gen_cpu = generated.detach().cpu()
        gen_np = (gen_cpu.permute(0, 2, 3, 1).numpy() * 255).astype(np.uint8)
        pil_images = [Image.fromarray(img) for img in gen_np]

        inputs = self.seg_processor(images=pil_images, return_tensors="pt").to(self.device)
        with torch.no_grad():
            outputs = self.seg_model(**inputs)
            logits = F.interpolate(outputs.logits, size=generated.shape[-2:], mode="bilinear", align_corners=False)
            pred_mask = torch.argmax(logits, dim=1)

        target_mask = self._rgb_to_sem_id(ground_truth_sem)

        # mIoU
        return self.miou_metric(pred_mask, target_mask).item()

    def compute_fid(self, generated, ground_truth):
        def get_feats(imgs):
            x = F.interpolate(imgs, size=(299, 299), mode='bilinear', align_corners=False)
            x = (x - 0.5) * 2
            return self.inception(x).detach().cpu().numpy()
        feat_gen = get_feats(generated)
        feat_gt = get_feats(ground_truth)
        mu1, sigma1 = np.mean(feat_gen, axis=0), np.cov(feat_gen, rowvar=False)
        mu2, sigma2 = np.mean(feat_gt, axis=0), np.cov(feat_gt, rowvar=False)
        diff = mu1 - mu2
        covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
        if np.iscomplexobj(covmean): covmean = covmean.real
        return float(diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean))

    def evaluate_batch(self, generated, ground_truth, semantic_map=None):
        res = {}
        gen = torch.clamp(generated, 0, 1)
        gt = torch.clamp(ground_truth, 0, 1)

        gen_np = gen.cpu().numpy().transpose(0, 2, 3, 1)
        gt_np = gt.cpu().numpy().transpose(0, 2, 3, 1)

        res['psnr'] = float(np.mean([psnr(gt_np[i], gen_np[i], data_range=1.0) for i in range(len(gen_np))]))
        res['ssim'] = float(np.mean([ssim(gt_np[i], gen_np[i], channel_axis=2, data_range=1.0) for i in range(len(gen_np))]))

        g_norm = gen*2-1
        t_norm = gt*2-1
        res['lpips'] = float(self.lpips_fn(g_norm.to(self.device), t_norm.to(self.device)).mean().item())

        res['mIoU'] = self.compute_miou(gen, semantic_map)

        return res

def evaluate_model(unet, resampler, vae, image_encoder, scheduler, dataloader, accelerator, output_dir, img_size, device, use_inpainting=False):
    evaluator = EvaluationMetrics(device=device, seg_model_id="./segformer_finetuned_change_bridge")

    unet.eval()
    resampler.eval()
    vae.eval()
    image_encoder.eval()

    all_metrics = {"psnr": [], "ssim": [], "lpips": [], "mIoU": []}

    all_gen_imgs = []
    all_gt_imgs = []

    for batch in tqdm(dataloader, desc="Generating & Scoring"):
        # batch to device
        pre_imgs = batch["pre"].to(device)
        sem_imgs = batch["sem"].to(device)
        post_imgs = batch["post"].to(device) # Ground Truth

        with torch.no_grad():
            pre_01 = (pre_imgs + 1.0) / 2.0
            pre_features = image_encoder(pre_01)
            encoder_hidden_states = resampler(pre_features)

            latent_size = img_size // 8

            if use_inpainting:
                source_latents = vae.encode(pre_imgs.to(dtype=vae.dtype)).latent_dist.sample() * vae.config.scaling_factor

                # mask from semantic map (white/keep)
                mask_1ch = (sem_imgs.mean(dim=1, keepdim=True)>0.5).float()
                mask_latents = F.interpolate(mask_1ch, size=source_latents.shape[-2:], mode="nearest")
                latents = torch.randn_like(source_latents)
            else:
                latents = torch.randn((pre_imgs.size(0), 4, latent_size, latent_size), device=device, dtype=unet.dtype)

            scheduler.set_timesteps(50)

            # denoising
            for t in scheduler.timesteps:
                sem_latents = F.interpolate(sem_imgs, size=latents.shape[-2:], mode="nearest")
                unet_in = torch.cat([latents, sem_latents], dim=1)

                noise_pred = unet(unet_in, t, encoder_hidden_states=encoder_hidden_states).sample
                latents_generated = scheduler.step(noise_pred, t, latents).prev_sample

                if use_inpainting:
                    # in the inpainting, the "keep" areas should look like the noisy source image at timestep t
                    noise_source = torch.randn_like(source_latents)
                    source_noisy = scheduler.add_noise(source_latents, noise_source, t.unsqueeze(0))
                    latents = (source_noisy*mask_latents)+(latents_generated*(1-mask_latents))
                else:
                    latents = latents_generated

            # decoding
            latents = latents / vae.config.scaling_factor
            gen_imgs = vae.decode(latents).sample

            # normalizing to [0, 1] for evaluation
            gen_imgs = (gen_imgs / 2 + 0.5).clamp(0, 1)
            gt_imgs_01 = (post_imgs / 2 + 0.5).clamp(0, 1)
            sem_imgs_01 = (sem_imgs / 2 + 0.5).clamp(0, 1)

        batch_res = evaluator.evaluate_batch(gen_imgs, gt_imgs_01, semantic_map=sem_imgs_01)

        for k, v in batch_res.items():
            all_metrics[k].append(v)

        all_gen_imgs.append(gen_imgs.cpu())
        all_gt_imgs.append(gt_imgs_01.cpu())

    final_results = {k + "_mean": np.mean(v) for k, v in all_metrics.items()}

    full_gen = torch.cat(all_gen_imgs, dim=0).to(device)
    full_gt = torch.cat(all_gt_imgs, dim=0).to(device)
    if full_gen.shape[0] > 16:
        fid_score = evaluator.compute_fid(full_gen, full_gt)
        final_results['fid'] = fid_score
    else:
        final_results['fid'] = -1.0
        print("Skipping FID, too few samples.")

    return final_results

def run_quantitative_analysis():
    CHECKPOINT_PATH = "/content/second_model_output/checkpoint-49"
    DATA_DIR = "./SECOND_dataset"
    IMG_SIZE = 224
    BATCH_SIZE = 8

    accelerator = Accelerator(mixed_precision="fp16")
    device = accelerator.device

    test_dataset = SECONDDataset(DATA_DIR, split="test", img_size=IMG_SIZE)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_dataloader = accelerator.prepare(test_dataloader)

    unet, resampler, vae, image_encoder = load_checkpoint(CHECKPOINT_PATH, device)
    noise_scheduler = DDPMScheduler.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", subfolder="scheduler")

    metrics = evaluate_model(
        unet=unet,
        resampler=resampler,
        vae=vae,
        image_encoder=image_encoder,
        scheduler=noise_scheduler,
        dataloader=test_dataloader,
        accelerator=accelerator,
        output_dir=CHECKPOINT_PATH,
        img_size=IMG_SIZE,
        device=device
    )


    print("Final Result Report")
    print(f"PSNR: {metrics.get('psnr_mean', 0):.2f}")
    print(f"SSIM: {metrics.get('ssim_mean', 0):.4f}")
    print(f"LPIPS: {metrics.get('lpips_mean', 0):.4f}")
    print(f"mIoU: {metrics.get('mIoU_mean', 0) * 100:.2f}%")
    print(f"FID: {metrics.get('fid', -1):.2f}")

# run this evaluation if working with a saved checkpoint from the previous training of the model, else run main first
"""
if __name__ == "__main__":
    run_quantitative_analysis()
"""

In [ ]:
def main():
    pretrained_model_name_or_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    data_dir = DATA_DIR
    output_dir = "second_model_output"
    train_batch_size = 16
    num_epochs = 30
    learning_rate = 1e-4
    img_size = 224
    gradient_accumulation_steps = 1
    seed = 42

    args = SimpleNamespace()
    args.pretrained_model_name_or_path = pretrained_model_name_or_path
    args.data_dir = data_dir
    args.output_dir = output_dir
    args.train_batch_size = train_batch_size
    args.num_epochs = num_epochs
    args.learning_rate = learning_rate
    args.img_size = img_size
    args.gradient_accumulation_steps = gradient_accumulation_steps
    args.seed = seed

    accelerator = Accelerator(
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        mixed_precision="fp16"
    )
    set_seed(args.seed)


    # VAE
    vae = AutoencoderKL.from_pretrained(args.pretrained_model_name_or_path, subfolder="vae")
    vae.requires_grad_(False)

    # UNet
    unet = UNet2DConditionModel.from_pretrained(args.pretrained_model_name_or_path, subfolder="unet")

    # modified UNet input to accept Semantic Map (3 channels) + Latents (4 channels) = 7 channels
    # semantic map is concatenated to the noisy latents
    with torch.no_grad():
        old_conv = unet.conv_in
        new_conv = nn.Conv2d(4 + 3, old_conv.out_channels, kernel_size=old_conv.kernel_size, padding=old_conv.padding)
        new_conv.weight[:, :4, :, :] = old_conv.weight
        new_conv.weight[:, 4:, :, :] = torch.zeros_like(new_conv.weight[:, 4:, :, :])
        new_conv.bias = old_conv.bias
        unet.conv_in = new_conv

    # image encoder (DinoV2)
    image_encoder = FrozenDinoV2Encoder()
    image_encoder.freeze()

    # Resampler (CoSampler adaptation)
    # DinoV2 output dim is 1024. UNet cross attention dim is usually 768 for SD 1.5. Resampler used for fixing dimension mismatch.
    resampler = ImageResampler(
        dim=1024,
        depth=4,
        dim_head=64,
        heads=16,
        num_queries=8,
        embedding_dim=1024, # DinoV2 dim
        output_dim=unet.config.cross_attention_dim, # UNet context dim
    )


    noise_scheduler = DDPMScheduler.from_pretrained(args.pretrained_model_name_or_path, subfolder="scheduler")

    # training UNet and Resampler
    params_to_optimize = list(unet.parameters()) + list(resampler.parameters())
    optimizer = torch.optim.AdamW(params_to_optimize, lr=args.learning_rate)

    dataset = SECONDDataset(args.data_dir, split="train", img_size=args.img_size)
    dataloader = DataLoader(dataset, batch_size=args.train_batch_size, shuffle=True, num_workers=4)

    # validation set
    val_dataset = SECONDDataset(args.data_dir, split="validation", img_size=args.img_size)
    if len(val_dataset) == 0:
        # subset of train for validation
        indices = torch.randperm(len(dataset))[:16]
        val_dataset = torch.utils.data.Subset(dataset, indices)

    val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=True, num_workers=4)

    unet, resampler, optimizer, dataloader, val_dataloader = accelerator.prepare(
        unet, resampler, optimizer, dataloader, val_dataloader
    )
    vae.to(accelerator.device)
    image_encoder.to(accelerator.device)

    # training
    global_step = 0
    for epoch in range(args.num_epochs):
        unet.train()
        resampler.train()

        progress_bar = tqdm(total=len(dataloader), disable=not accelerator.is_local_main_process)
        progress_bar.set_description(f"Epoch {epoch}")

        for step, batch in enumerate(dataloader):
            with accelerator.accumulate(unet, resampler):
               # encoding post image(target) -> latents
                post_imgs = batch["post"].to(dtype=vae.dtype)
                latents = vae.encode(post_imgs).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

                # sampling noise
                noise = torch.randn_like(latents)
                bsz = latents.shape[0]
                timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=latents.device)
                timesteps = timesteps.long()

                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                # pre image -> resampler -> cross attention context
                pre_imgs = batch["pre"] # (B, 3, H, W)
                pre_imgs_01 = (pre_imgs+1.0)/2.0

                with torch.no_grad():
                    pre_features = image_encoder(pre_imgs_01) # (B, 257, 1024)

                encoder_hidden_states = resampler(pre_features) # (B, 8, 768)

                # semantic map -> concatenate to noisy latents
                sem_imgs = batch["sem"] # (B, 3, H, W)
                sem_latents = F.interpolate(sem_imgs, size=noisy_latents.shape[-2:], mode="nearest")

                unet_input = torch.cat([noisy_latents, sem_latents], dim=1)

                # predicting noise
                model_pred = unet(unet_input, timesteps, encoder_hidden_states=encoder_hidden_states).sample

                loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")

                accelerator.backward(loss)
                optimizer.step()
                optimizer.zero_grad()

            if global_step % 100 == 0:
                if accelerator.is_main_process:
                    log_validation(
                        unet=accelerator.unwrap_model(unet),
                        resampler=accelerator.unwrap_model(resampler),
                        vae=vae,
                        image_encoder=image_encoder,
                        scheduler=noise_scheduler,
                        dataloader=val_dataloader,
                        accelerator=accelerator,
                        epoch=epoch,
                        step=global_step,
                        output_dir=args.output_dir,
                        img_size=args.img_size
                    )

            progress_bar.update(1)
            progress_bar.set_postfix(loss=loss.item())
            global_step += 1

        if epoch % 5 == 0:
            metrics = evaluate_model(
                unet=accelerator.unwrap_model(unet),
                resampler=accelerator.unwrap_model(resampler),
                vae=vae,
                image_encoder=image_encoder,
                scheduler=noise_scheduler,
                dataloader=val_dataloader, # ** we used Validation Set to evaluate our model performance on the way **
                accelerator=accelerator,
                output_dir=args.output_dir,
                img_size=args.img_size,
                device=accelerator.device
            )

    test_dataset = SECONDDataset(args.data_dir, split="test", img_size=args.img_size)

    test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=4)
    test_dataloader = accelerator.prepare(test_dataloader)

    final_metrics = evaluate_model(
        unet=accelerator.unwrap_model(unet),
        resampler=accelerator.unwrap_model(resampler),
        vae=vae,
        image_encoder=image_encoder,
        scheduler=noise_scheduler,
        dataloader=test_dataloader,
        accelerator=accelerator,
        output_dir=args.output_dir,
        img_size=args.img_size,
        device=accelerator.device,
        use_inpainting=True  # can be set to false, but inpaintined results are better
    )

    print("Final scores:", final_metrics)
    # saving checkpoint
    if accelerator.is_main_process:
        save_path = os.path.join(args.output_dir, f"checkpoint-{epoch}")
        os.makedirs(save_path, exist_ok=True)
        unet_unwrapped = accelerator.unwrap_model(unet)
        resampler_unwrapped = accelerator.unwrap_model(resampler)

        unet_unwrapped.save_pretrained(os.path.join(save_path, "unet"))
        torch.save(resampler_unwrapped.state_dict(), os.path.join(save_path, "resampler.pt"))

if __name__ == "__main__":
    main()


In [ ]:
import matplotlib.pyplot as plt
def generate_qualitative_samples(checkpoint_path, output_image_name="qualitative_grid.png"):
    DATA_DIR = "./SECOND_dataset"
    IMG_SIZE = 224
    NUM_SAMPLES = 4

    accelerator = Accelerator(mixed_precision="fp16")
    device = accelerator.device

    test_dataset = SECONDDataset(DATA_DIR, split="test", img_size=IMG_SIZE)
    if len(test_dataset) == 0:
        test_dataset = SECONDDataset(DATA_DIR, split="validation", img_size=IMG_SIZE)

    # to get random samples each time running this
    dataloader = DataLoader(test_dataset, batch_size=NUM_SAMPLES, shuffle=True)
    batch = next(iter(dataloader)) # one batch at a time

    unet, resampler, vae, image_encoder = load_checkpoint(checkpoint_path, device)
    noise_scheduler = DDPMScheduler.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", subfolder="scheduler")

    print("Generating images")
    unet.eval()
    resampler.eval()

    pre_imgs = batch["pre"].to(device)
    sem_imgs = batch["sem"].to(device)
    post_imgs = batch["post"].to(device) # Ground Truth

    with torch.no_grad():
        pre_imgs_01 = (pre_imgs + 1.0) / 2.0
        pre_features = image_encoder(pre_imgs_01)
        encoder_hidden_states = resampler(pre_features)

        latent_size = IMG_SIZE // 8
        latents = torch.randn((NUM_SAMPLES, 4, latent_size, latent_size), device=device, dtype=unet.dtype)

        # diffusion
        noise_scheduler.set_timesteps(50)
        for t in noise_scheduler.timesteps:
            sem_latents = F.interpolate(sem_imgs, size=latents.shape[-2:], mode="nearest")
            unet_input = torch.cat([latents, sem_latents], dim=1)
            noise_pred = unet(unet_input, t, encoder_hidden_states=encoder_hidden_states).sample
            latents = noise_scheduler.step(noise_pred, t, latents).prev_sample

        # decoding
        generated_images = vae.decode(latents / vae.config.scaling_factor).sample


    def denorm(x):
        return (x / 2 + 0.5).clamp(0, 1).cpu().permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(NUM_SAMPLES, 4, figsize=(16, 4 * NUM_SAMPLES))

    cols = ["Pre-Disaster (Input)", "Semantic Map (Input)", "Ground Truth (Post)", "Generated (Ours)"]
    for ax, col in zip(axes[0], cols):
        ax.set_title(col, fontsize=14, fontweight='bold')

    for i in range(NUM_SAMPLES):
        axes[i, 0].imshow(denorm(pre_imgs[i]))
        axes[i, 0].axis('off')

        sem_vis = denorm(sem_imgs[i])
        if sem_vis.shape[-1] == 3:
            sem_vis = sem_vis.mean(axis=-1)
        axes[i, 1].imshow(sem_vis, cmap='gray')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(denorm(post_imgs[i]))
        axes[i, 2].axis('off')

        axes[i, 3].imshow(denorm(generated_images[i]))
        axes[i, 3].axis('off')

    plt.tight_layout()
    plt.savefig(output_image_name, dpi=300, bbox_inches='tight')
    plt.show()

CHECKPOINT = "/content/second_model_output/checkpoint-49" # last epoch (we are able to train 50 epoch at max)
generate_qualitative_samples(CHECKPOINT)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import os
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler
from torch.utils.data import DataLoader
from accelerate import Accelerator

def generate_qualitative_samples_inpainting(checkpoint_path, output_image_name="qualitative_grid_inpainting.png"):
    DATA_DIR = "./SECOND_dataset"
    IMG_SIZE = 224
    NUM_SAMPLES = 4

    accelerator = Accelerator(mixed_precision="fp16")
    device = accelerator.device

    test_dataset = SECONDDataset(DATA_DIR, split="test", img_size=IMG_SIZE)
    if len(test_dataset) == 0:
        test_dataset = SECONDDataset(DATA_DIR, split="validation", img_size=IMG_SIZE)

    dataloader = DataLoader(test_dataset, batch_size=NUM_SAMPLES, shuffle=True)
    batch = next(iter(dataloader))

    unet, resampler, vae, image_encoder = load_checkpoint(checkpoint_path, device)
    noise_scheduler = DDPMScheduler.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", subfolder="scheduler")

    print("Generating images with inpainting logic")
    unet.eval()
    resampler.eval()

    pre_imgs = batch["pre"].to(device)
    sem_imgs = batch["sem"].to(device)
    post_imgs = batch["post"].to(device)

    with torch.no_grad():
        pre_imgs_01 = (pre_imgs + 1.0) / 2.0
        pre_features = image_encoder(pre_imgs_01)
        encoder_hidden_states = resampler(pre_features)

        source_latents = vae.encode(pre_imgs.to(dtype=vae.dtype)).latent_dist.sample() * vae.config.scaling_factor

        mask_1ch = (sem_imgs.mean(dim=1, keepdim=True) > 0.5).float()

        mask_latents = F.interpolate(mask_1ch, size=source_latents.shape[-2:], mode="nearest")

        # diffusion
        latents = torch.randn_like(source_latents)
        noise_scheduler.set_timesteps(50)

        for t in noise_scheduler.timesteps:
            #  noise prediction
            sem_latents_input = F.interpolate(sem_imgs, size=latents.shape[-2:], mode="nearest")
            unet_input = torch.cat([latents, sem_latents_input], dim=1)

            noise_pred = unet(unet_input, t, encoder_hidden_states=encoder_hidden_states).sample

            latents_generated = noise_scheduler.step(noise_pred, t, latents).prev_sample

            # constraint
            noise_source = torch.randn_like(source_latents)
            source_noisy = noise_scheduler.add_noise(source_latents, noise_source, t.unsqueeze(0))

            latents = (source_noisy * mask_latents) + (latents_generated * (1 - mask_latents))
        generated_images = vae.decode(latents / vae.config.scaling_factor).sample

    def denorm(x):
        return (x / 2 + 0.5).clamp(0, 1).cpu().permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(NUM_SAMPLES, 4, figsize=(16, 4 * NUM_SAMPLES))
    cols = ["Pre-Disaster (Input)", "Semantic Map (White=Keep)", "Ground Truth", "Generated (Inpainted)"]
    for ax, col in zip(axes[0], cols):
        ax.set_title(col, fontsize=14, fontweight='bold')

    for i in range(NUM_SAMPLES):
        axes[i, 0].imshow(denorm(pre_imgs[i]))
        axes[i, 1].imshow(denorm(sem_imgs[i]).mean(axis=-1), cmap='gray')
        axes[i, 2].imshow(denorm(post_imgs[i]))
        axes[i, 3].imshow(denorm(generated_images[i]))
        for ax in axes[i]: ax.axis('off')

    plt.tight_layout()
    plt.savefig(output_image_name, dpi=300, bbox_inches='tight')
    plt.show()

CHECKPOINT = "/content/second_model_output/checkpoint-49"
generate_qualitative_samples_inpainting(CHECKPOINT)

In [ ]:
import torch
from torch.utils.data import DataLoader, ConcatDataset
from tqdm.auto import tqdm

def run_fid_with_combined_reference(checkpoint_path):
    DATA_DIR = "./SECOND_dataset"
    IMG_SIZE = 224
    BATCH_SIZE = 16

    accelerator = Accelerator(mixed_precision="fp16")
    device = accelerator.device

    test_dataset = SECONDDataset(DATA_DIR, split="test", img_size=IMG_SIZE)

    # Reference: train + validation (for Real Ground Truth to check image realisticity with high # samples)
    train_dataset = SECONDDataset(DATA_DIR, split="train", img_size=IMG_SIZE)
    val_dataset = SECONDDataset(DATA_DIR, split="validation", img_size=IMG_SIZE)

    ref_dataset = ConcatDataset([train_dataset, val_dataset])

    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    ref_loader = DataLoader(ref_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    test_loader, ref_loader = accelerator.prepare(test_loader, ref_loader)

    unet, resampler, vae, image_encoder = load_checkpoint(checkpoint_path, device)
    noise_scheduler = DDPMScheduler.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", subfolder="scheduler")

    generated_images = []

    unet.eval()
    resampler.eval()

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Generating"):
            pre_imgs = batch["pre"].to(device)
            sem_imgs = batch["sem"].to(device)

            # inpainting logic
            pre_imgs_01 = (pre_imgs+1.0)/2.0
            pre_features = image_encoder(pre_imgs_01)
            encoder_hidden_states = resampler(pre_features)

            source_latents = vae.encode(pre_imgs.to(dtype=vae.dtype)).latent_dist.sample() * vae.config.scaling_factor
            latents = torch.randn_like(source_latents)

            # white = keep
            mask_1ch = (sem_imgs.mean(dim=1, keepdim=True)>0.5).float()
            mask_latents = F.interpolate(mask_1ch, size=source_latents.shape[-2:], mode="nearest")

            noise_scheduler.set_timesteps(50)
            for t in noise_scheduler.timesteps:
                sem_latents = F.interpolate(sem_imgs, size=latents.shape[-2:], mode="nearest")
                unet_input = torch.cat([latents, sem_latents], dim=1)

                noise_pred = unet(unet_input, t, encoder_hidden_states=encoder_hidden_states).sample
                latents_gen = noise_scheduler.step(noise_pred, t, latents).prev_sample

                # constraint
                noise_source = torch.randn_like(source_latents)
                source_noisy = noise_scheduler.add_noise(source_latents, noise_source, t.unsqueeze(0))
                latents = (source_noisy * mask_latents) + (latents_gen * (1 - mask_latents))

            imgs = vae.decode(latents / vae.config.scaling_factor).sample
            imgs = (imgs / 2 + 0.5).clamp(0, 1)
            generated_images.append(imgs.cpu())

    all_generated = torch.cat(generated_images, dim=0)

    real_images = []

    with torch.no_grad():
        for batch in tqdm(ref_loader, desc="Collecting Real"):
            real = batch["post"]
            real = (real / 2 + 0.5).clamp(0, 1)
            real_images.append(real.cpu())

    all_real = torch.cat(real_images, dim=0)

    evaluator = EvaluationMetrics(device=device)
    fid_score = evaluator.compute_fid(all_generated, all_real)

    print(f"FID score: {fid_score:.2f}")

CHECKPOINT = "/content/second_model_output/checkpoint-49"
run_fid_with_combined_reference(CHECKPOINT)

In [ ]:
!mv /content/second_model_output/checkpoint-49 /content/drive/MyDrive/cs559-project/

**Base U-net Model for comparison**

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as TF
from PIL import Image
from diffusers import AutoencoderKL, UNet2DModel, DDPMScheduler
from accelerate import Accelerator
from accelerate.utils import set_seed
from tqdm.auto import tqdm
import numpy as np

class Config:
    img_size = 256
    batch_size = 8
    num_epochs = 100
    learning_rate = 1e-4
    gradient_accumulation_steps = 1
    mixed_precision = "fp16"
    output_dir = "base_model_output"
    save_image_epochs = 5
    save_model_epochs = 20
    seed = 42


def log_validation_base(unet, vae, scheduler, dataloader, accelerator, epoch, step, output_dir, img_size):
    print(f"Running validation at epoch {epoch} step {step}...")
    unet.eval()

    try:
        batch = next(iter(dataloader))
    except StopIteration:
        batch = next(iter(dataloader))

    pre_imgs = batch["pre"].to(accelerator.device)
    sem_imgs = batch["sem"].to(accelerator.device)
    post_imgs = batch["post"].to(accelerator.device)

    with torch.no_grad():
        pre_latents = vae.encode(pre_imgs).latent_dist.sample()
        pre_latents = pre_latents * vae.config.scaling_factor

    latent_size = img_size // 8


    latents = torch.randn(
        (pre_imgs.shape[0], 4, latent_size, latent_size),
        device=accelerator.device,
        dtype=unet.dtype
    )

    scheduler.set_timesteps(50)

    for t in tqdm(scheduler.timesteps, disable=True):
        sem_latents = F.interpolate(sem_imgs, size=latents.shape[-2:], mode="nearest")
        unet_input = torch.cat([latents, sem_latents, pre_latents], dim=1)

        with torch.no_grad():
            noise_pred = unet(unet_input, t).sample

        latents = scheduler.step(noise_pred, t, latents).prev_sample

    latents = latents / vae.config.scaling_factor
    with torch.no_grad():
        images = vae.decode(latents).sample

    images = (images / 2 + 0.5).clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()
    pre_imgs_disp = (pre_imgs / 2 + 0.5).clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()
    post_imgs_disp = (post_imgs / 2 + 0.5).clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()
    sem_imgs_disp = (sem_imgs / 2 + 0.5).clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()

    save_dir = os.path.join(output_dir, "validation")
    os.makedirs(save_dir, exist_ok=True)

    for i in range(images.shape[0]):
        combined = np.concatenate([pre_imgs_disp[i], sem_imgs_disp[i], post_imgs_disp[i], images[i]], axis=1)
        combined_img = Image.fromarray((combined * 255).astype(np.uint8))
        combined_img.save(os.path.join(save_dir, f"epoch_{epoch}_sample_{i}.png"))

    unet.train()

def main():
    config = Config()


    set_seed(config.seed)

    accelerator = Accelerator(
        mixed_precision=config.mixed_precision,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        log_with="tensorboard",
        project_dir=config.output_dir
    )

    if accelerator.is_main_process:
        os.makedirs(config.output_dir, exist_ok=True)
        # Seed bilgisini de loglayabilirsin
        print(f"Training initialized with seed: {config.seed}")

    vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
    vae.requires_grad_(False)

    unet = UNet2DModel(
        sample_size=config.img_size // 8,
        in_channels=11,
        out_channels=4,
        layers_per_block=2,
        block_out_channels=(128, 128, 256, 256, 512, 512),
        down_block_types=(
            "DownBlock2D", "DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "DownBlock2D"
        ),
        up_block_types=(
            "UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D"
        ),
    )

    noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
    optimizer = torch.optim.AdamW(unet.parameters(), lr=config.learning_rate)

    dataset = SECONDDataset(root_dir="./SECOND_dataset", split="train", img_size=config.img_size)
    val_dataset = SECONDDataset(root_dir="./SECOND_dataset", split="validation", img_size=config.img_size)


    def seed_worker(worker_id):
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        import random
        random.seed(worker_seed)

    g = torch.Generator()
    g.manual_seed(config.seed)

    train_dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=4,
        worker_init_fn=seed_worker,
        generator=g
    )

    val_dataloader = DataLoader(
        val_dataset,
        batch_size=4,
        shuffle=False,
        worker_init_fn=seed_worker,
        generator=g
    )

    unet, optimizer, train_dataloader = accelerator.prepare(
        unet, optimizer, train_dataloader
    )
    vae.to(accelerator.device)

    global_step = 0
    for epoch in range(config.num_epochs):
        unet.train()
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(unet):
                pre_imgs = batch["pre"].to(accelerator.device)
                post_imgs = batch["post"].to(accelerator.device)
                sem_imgs = batch["sem"].to(accelerator.device)

                with torch.no_grad():
                    latents = vae.encode(post_imgs).latent_dist.sample()
                    latents = latents * vae.config.scaling_factor

                    pre_latents = vae.encode(pre_imgs).latent_dist.sample()
                    pre_latents = pre_latents * vae.config.scaling_factor

                noise = torch.randn_like(latents)
                bsz = latents.shape[0]
                timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=latents.device)
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                sem_latents = F.interpolate(sem_imgs, size=latents.shape[-2:], mode="nearest")
                model_input = torch.cat([noisy_latents, sem_latents, pre_latents], dim=1)

                model_pred = unet(model_input, timesteps).sample
                loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")

                accelerator.backward(loss)
                optimizer.step()
                optimizer.zero_grad()

            global_step += 1

            if global_step % 100 == 0:
                accelerator.print(f"Epoch {epoch} | Step {global_step} | Loss: {loss.item()}")

        if epoch % config.save_image_epochs == 0:
            if accelerator.is_main_process:
                log_validation_base(unet, vae, noise_scheduler, val_dataloader, accelerator, epoch, global_step, config.output_dir, config.img_size)

        if epoch % config.save_model_epochs == 0:
             if accelerator.is_main_process:
                unet.save_pretrained(os.path.join(config.output_dir, f"checkpoint-{epoch}"))

if __name__ == "__main__":
    main()

In [ ]:
import shutil

source_path = "./base_model_output"
destination_path = "/content/drive/MyDrive/backup_base_model1"

if os.path.exists(source_path):
    print("Copying to drive")
    shutil.copytree(source_path, destination_path, dirs_exist_ok=True)
    print("saved")

Evaluate base model

In [ ]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as TF
from PIL import Image
import numpy as np
from tqdm.auto import tqdm
import evaluate
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
    TrainingArguments,
    Trainer
)
from diffusers import AutoencoderKL, UNet2DModel, DDPMScheduler
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
from torchmetrics.image import FrechetInceptionDistance
import lpips


DATA_DIR = "./SECOND_dataset"
OUTPUT_DIR_SEG = "./segformer_finetuned_judge"
BASE_MODEL_PATH = "./base_model_output"

IMG_SIZE = 256
NUM_CLASSES = 7
MODEL_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"

PALETTE = {
    (255, 255, 255): 0, (128, 128, 128): 1, (0, 128, 0): 2,
    (0, 255, 0): 3,     (0, 0, 255): 4,     (128, 0, 0): 5,
    (255, 0, 0): 6
}
def run_full_evaluation():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    subdirs = [x[0] for x in os.walk(BASE_MODEL_PATH)]
    checkpoints = sorted([x for x in subdirs if "checkpoint" in x], key=lambda x: int(x.split("-")[-1]))

    unet = UNet2DModel.from_pretrained(checkpoints[-1], use_safetensors=True).to(device).eval()
    vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device).eval()
    scheduler = DDPMScheduler(num_train_timesteps=1000)

    seg_model = SegformerForSemanticSegmentation.from_pretrained(OUTPUT_DIR_SEG).to(device).eval()
    seg_processor = SegformerImageProcessor.from_pretrained(OUTPUT_DIR_SEG)

    psnr = PeakSignalNoiseRatio(data_range=1.0).to(device)
    ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
    fid = FrechetInceptionDistance(feature=2048).to(device)
    lpips_fn = lpips.LPIPS(net='alex').to(device)
    metric_iou = evaluate.load("mean_iou")

    val_split = "validation" if os.path.exists(os.path.join(DATA_DIR, "validation")) else "val"
    test_ds = GenerationTestDataset(DATA_DIR, val_split, IMG_SIZE)
    loader = DataLoader(test_ds, batch_size=4, shuffle=False)

    total_lpips = 0

    for batch in tqdm(loader, desc="Evaluating"):
        pre = batch["pre"].to(device)
        post = batch["post"].to(device)
        sem = batch["sem"].to(device)

        with torch.no_grad():
            pre_latents = vae.encode(pre).latent_dist.sample() * vae.config.scaling_factor
            latents = torch.randn_like(pre_latents)
            scheduler.set_timesteps(50)
            for t in scheduler.timesteps:
                sem_latents = F.interpolate(sem, size=latents.shape[-2:], mode="nearest")
                inp = torch.cat([latents, sem_latents, pre_latents], dim=1)
                noise = unet(inp, t).sample
                latents = scheduler.step(noise, t, latents).prev_sample
            gen = vae.decode(latents / vae.config.scaling_factor).sample

        gen_01 = (gen / 2 + 0.5).clamp(0, 1)
        real_01 = (post / 2 + 0.5).clamp(0, 1)

        psnr.update(gen_01, real_01)
        ssim.update(gen_01, real_01)
        fid.update((real_01 * 255).to(torch.uint8), real=True)
        fid.update((gen_01 * 255).to(torch.uint8), real=False)
        total_lpips += lpips_fn(gen, post).sum().item()

        pil_imgs = [TF.ToPILImage()(g) for g in gen_01.cpu()]
        inputs = seg_processor(images=pil_imgs, return_tensors="pt").to(device)
        with torch.no_grad():
            outs = seg_model(**inputs)
            logits = F.interpolate(outs.logits, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
            pred_masks = logits.argmax(dim=1).cpu().numpy()

        img_denorm = (sem.cpu() / 2 + 0.5).clamp(0, 1) * 255
        img_denorm = img_denorm.permute(0, 2, 3, 1).numpy().astype(np.uint8)
        ref_masks = []
        for i in range(img_denorm.shape[0]):
            arr = img_denorm[i]
            mask = np.zeros(arr.shape[:2], dtype=np.int32)
            for col, cid in PALETTE.items():
                matches = np.all(np.abs(arr - col) < 30, axis=-1)
                mask[matches] = cid
            ref_masks.append(mask)

        metric_iou.add_batch(predictions=pred_masks, references=np.array(ref_masks))

    res_psnr = psnr.compute().item()
    res_ssim = ssim.compute().item()
    res_fid = fid.compute().item()
    res_lpips = total_lpips / len(test_ds)
    res_iou = metric_iou.compute(num_labels=NUM_CLASSES, ignore_index=255)

    print("Final Result Report")
    print(f"PSNR : {res_psnr:.4f}")
    print(f"SSIM : {res_ssim:.4f}")
    print(f"LPIPS : {res_lpips:.4f}")
    print(f"FID : {res_fid:.4f}")
    print(f"Mean IoU : {res_iou['mean_iou']:.4f}")
    print(f"Mean Accuracy : {res_iou['mean_accuracy']:.4f}")

# run this evaluation if working with a saved checkpoint from the previous training of the model, else run main first

if __name__ == "__main__":
    run_full_evaluation()